In [1]:
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/kitti3.zip /content/
!unzip -q -o /content/kitti3.zip -d /content/

yaml_text = """path: /content/kitti3
train: images/train
val: images/val
test: images/test
names:
  0: vehicle
  1: pedestrian
  2: cyclist
      """
with open("/content/kitti3/data.yaml", "w") as f:
  f.write(yaml_text)

DATA = "/content/kitti3/data.yaml"

!nvidia-smi | head -n 12


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wed Nov 11 13:17:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|          

In [2]:
import os, shutil
from ultralytics import YOLO

RUN = "/content/drive/MyDrive/driving-perception/runs/improved_yolo11s/weights"
os.makedirs("/content/export", exist_ok=True)
shutil.copy(f"{RUN}/best.pt", "/content/export/model.pt")

m = YOLO("/content/export/model.pt")
onnx_path = m.export(format="onnx", imgsz=640, simplify=True)
print("ONNX:", onnx_path)

os.makedirs("/content/drive/MyDrive/driving-perception/models", exist_ok=True)
shutil.copy("/content/export/model.pt", "/content/drive/MyDrive/driving-perception/models/model.pt")
shutil.copy(onnx_path, "/content/drive/MyDrive/driving-perception/models/model.onnx")

Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11s summary (fused): 100 layers, 9,413,961 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from '/content/export/model.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (18.3 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 3.4s, saved as '/content/export/model.onnx' (36.2 MB)

Export complete (4.6s)
Results saved to /content/export/model.onnx
Predict:         yolo predict task=detect model=/content/export/model.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/export/model.onnx imgsz=640 data=/content/kitti3/data.yaml 
Visualize:       https://netron.app
ONNX: /content/export/model.onnx


'/content/drive/MyDrive/driving-perception/models/model.onnx'

In [3]:
import os, shutil
from ultralytics import YOLO

RUN = "/content/drive/MyDrive/driving-perception/runs/improved_yolo11s/weights"
os.makedirs("/content/export", exist_ok=True)
shutil.copy(f"{RUN}/best.pt", "/content/export/model.pt")

m = YOLO("/content/export/model.pt")
onnx_path = m.export(format="onnx", imgsz=640, simplify=True)
print("ONNX:", onnx_path)

os.makedirs("/content/drive/MyDrive/driving-perception/models", exist_ok=True)
shutil.copy("/content/export/model.pt", "/content/drive/MyDrive/driving-perception/models/model.pt")
shutil.copy(onnx_path, "/content/drive/MyDrive/driving-perception/models/model.onnx")

Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11s summary (fused): 100 layers, 9,413,961 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from '/content/export/model.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (18.3 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 2.5s, saved as '/content/export/model.onnx' (36.2 MB)

Export complete (3.4s)
Results saved to /content/export/model.onnx
Predict:         yolo predict task=detect model=/content/export/model.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/export/model.onnx imgsz=640 data=/content/kitti3/data.yaml 
Visualize:       https://netron.app
ONNX: /content/export/model.onnx


'/content/drive/MyDrive/driving-perception/models/model.onnx'

In [4]:
m = YOLO("/content/export/model.pt")
engine_path = m.export(format="engine", imgsz=640, half=True, device=0)
print("TensorRT engine:", engine_path)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,961 parameters, 0 gradients, 21.4 GFLOPs

PyTorch: starting from '/content/export/model.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (18.3 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 3.5s, saved as '/content/export/model.onnx' (36.2 MB)

TensorRT: starting export with TensorRT 11.4.0.106...
TensorRT: converting ONNX to FP16 mixed precision with ModelOpt AutoCast...
2026-11-11 13:18:00,875 - [modelopt][onnx] - WARNING - Shared constants were detected and duplicated accordingly.


2026-11-11 13:18:01,841 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-11-11 13:18:02,318 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


2026-11-11 13:18:05,244 - [modelopt][onnx] - WARNING - Did not find  in value info map! Assuming not castable


2026-11-11 13:18:05,352 - [modelopt][onnx] - WARNING - Some initializers contain values smaller than smallest fp16 value, values will be replaced with 6.0e-08.


2026-11-11 13:18:07,420 - [modelopt][onnx] - INFO - Converted 320/320 nodes (100.00%) to fp16


INFO:modelopt.onnx.autocast:Converted 320/320 nodes (100.00%) to fp16


TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 7, 8400) DataType.FLOAT
TensorRT: building FP16 engine as /content/export/model.engine
TensorRT: export success ✅ 173.6s, saved as '/content/export/model.engine' (90.8 MB)

Export complete (178.6s)
Results saved to /content/export/model.engine
Predict:         yolo predict task=detect model=/content/export/model.engine imgsz=640 quantize=16
Validate:        yolo val task=detect model=/content/export/model.engine imgsz=640 data=/content/kitti3/data.yaml quantize=16
Visualize:       https://netron.app
TensorRT engine: /content/export/model.engine


In [5]:
import os, glob, time, json
import numpy as np, pandas as pd, torch
from ultralytics import YOLO

TEST_DIR = "/content/kitti3/images/test"
paths = sorted(glob.glob(f"{TEST_DIR}/*.png") + glob.glob(f"{TEST_DIR}/*.jpg"))[:300]
gpu = torch.cuda.get_device_name(0)
print(len(paths), "test images on", gpu)


def bench(weights, label):
  model = YOLO(weights, task="detect")
  for p in paths[:20]:
    model.predict(p, imgsz=640, device=0, verbose=False)
  inf, t0 = [], time.time()
  for p in paths:
    r = model.predict(p, imgsz=640, device=0, verbose=False)[0]
    inf.append(r.speed["inference"])
  wall = time.time() - t0
  try:
    m = model.val(data=DATA, split="test", imgsz=640, batch=1, device=0, verbose=False, plots=False)
    map50, map5095 = float(m.box.map50), float(m.box.map)
  except Exception as e:
    print("val failed for", label, repr(e)[:200])
    map50 = map5095 = None
  return {
      "format": label, "gpu": gpu, "images_timed": len(paths),
      "mean_inference_ms": round(float(np.mean(inf)), 2),
      "median_inference_ms": round(float(np.median(inf)), 2),
      "fps_from_median_inference": round(1000 / float(np.median(inf)), 1),
      "end_to_end_ms_per_image": round(wall / len(paths) * 1000, 2),
      "test_mAP50": None if map50 is None else round(map50, 3),
      "test_mAP50-95": None if map5095 is None else round(map5095, 3),
  }
results = []
for weights, label in [("/content/export/model.pt", "PyTorch FP32"),
                       ("/content/export/model.onnx", "ONNX"),
                       ("/content/export/model.engine", "TensorRT FP16")]:
  try:
    results.append(bench(weights, label))
  except Exception as e:
    print("FAILED:", label, "->", repr(e)[:300])

os.makedirs("/content/drive/MyDrive/driving-perception/benchmark", exist_ok=True)
df = pd.DataFrame(results)
df.to_csv("/content/drive/MyDrive/driving-perception/benchmark/benchmark_results.csv", index=False)
with open("/content/drive/MyDrive/driving-perception/benchmark/benchmark_results.json", "w") as f:
  json.dump(results, f, indent=2)
print(df.to_string(index=False))



300 test images on Tesla T4
Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,961 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1037.0±443.1 MB/s, size: 56.3 KB)
val: Scanning /content/kitti3/labels/test.cache... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 196.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1496/1496 48.0it/s 31.2s
                   all       1496       7920      0.865      0.742      0.831      0.556
Speed: 0.7ms preprocess, 13.4ms inference, 0.0ms loss, 1.3ms postprocess per image
Loading /content/export/model.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CUDAExecutionProvider
Ultralytics 8.4.210 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/export/model.onnx for ONNX Runtime inference...
Using ONNX Run